In [20]:
import os
import yaml
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import gc
import json

# ============================================================
# 10_modeling_hourly.ipynb
#
# [목적]
#   hourly aggregation 전처리 데이터로 RF 학습
#   - 피처: cpu, memory, duration, hour
#   - 타겟: power_w
#   - 기존 모델과 별도 파일명으로 저장
# ============================================================

CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])
MODEL_PATH = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")

# hourly 전처리 파일 로드
# hourly 전처리 파일 로드
hourly_file = os.path.join(PROCESSED_PATH, "instance_usage_hourly_processed.parquet")
df = pd.read_parquet(hourly_file)
print(f"Hourly rows: {len(df):,}")
print(f"\n[duration 통계]")
print(df['duration'].describe())
print(f"\n[power_w 통계]")
print(df['power_w'].describe())

Hourly rows: 1,454,606

[duration 통계]
count    1.454606e+06
mean     3.265929e+03
std      1.804283e+03
min      1.000000e+00
25%      3.065000e+03
50%      3.600000e+03
75%      3.600000e+03
max      2.353400e+04
Name: duration, dtype: float64

[power_w 통계]
count    1.454606e+06
mean     2.022671e+02
std      2.384973e+00
min      2.000000e+02
25%      2.009867e+02
50%      2.016166e+02
75%      2.027807e+02
max      3.037326e+02
Name: power_w, dtype: float64


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
# ============================================================
# 피처 / 타겟 설정
# 피처: cpu, memory, duration, hour (전부!)
# 타겟: energy_kwh
# ============================================================
features = ["cpu", "memory", "duration", "hour"]
target   = "energy_kwh"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)
del df; gc.collect()

print(f"\nFeatures : {features}")
print(f"Target   : {target}")
print(f"Train    : {len(X_train):,}")
print(f"Test     : {len(X_test):,}")


Features : ['cpu', 'memory', 'duration', 'hour']
Target   : energy_kwh
Train    : 1,163,684
Test     : 290,922


In [22]:
# ============================================================
# RF 학습
# ============================================================
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=config['model']['random_state'],
    n_jobs=-1
)
rf.fit(X_train, y_train)
print(f"\nRF trained!")
print(f"feature_names_in_: {rf.feature_names_in_}")


RF trained!
feature_names_in_: ['cpu' 'memory' 'duration' 'hour']


In [23]:
# ============================================================
# 평가
# ============================================================
y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100

print(f"\n[RF (hourly) 성능]")
print(f"  RMSE : {rmse:.8f} kWh")
print(f"  MAE  : {mae:.8f} kWh")
print(f"  R2   : {r2:.8f}")
print(f"  MAPE : {mape:.4f}%")

# Feature Importance
importance = pd.DataFrame({
    'feature'   : features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\n[Feature Importance]")
print(importance.to_string(index=False))



[RF (hourly) 성능]
  RMSE : 0.00038850 kWh
  MAE  : 0.00013112 kWh
  R2   : 0.99998528
  MAPE : 0.1077%

[Feature Importance]
 feature   importance
duration 9.994943e-01
     cpu 5.009822e-04
  memory 3.876916e-06
    hour 8.006353e-07


In [24]:
# ============================================================
# 모델 저장 (기존 모델과 별도 파일명)
# ============================================================
model_file = os.path.join(MODEL_PATH, config['model']['model_names']['rf_hourly'])
with open(model_file, 'wb') as f:
    pickle.dump(rf, f)
print(f"\nSaved: {model_file} ({os.path.getsize(model_file)/(1024**2):.1f} MB)")
print(f"Saved feature_names_in_: {rf.feature_names_in_}")




Saved: /content/drive/MyDrive/EcoTracing/models/energy_model_rf_hourly.pkl (227.6 MB)
Saved feature_names_in_: ['cpu' 'memory' 'duration' 'hour']


In [25]:

with open(model_file, 'rb') as f:
    loaded_rf = pickle.load(f)

print(f"[DEBUG] rf.feature_names_in_: {loaded_rf.feature_names_in_}")

[DEBUG] rf.feature_names_in_: ['cpu' 'memory' 'duration' 'hour']


In [26]:
# ============================================================
# 결과 저장
# ============================================================
result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json_hourly'])

results = {
    'method'   : 'RandomForest (hourly aggregation)',
    'features' : features,
    'target'   : target,
    'rf_params': {
        'n_estimators'     : 100,
        'max_depth'        : 15,
        'min_samples_split': 10,
        'min_samples_leaf' : 5,
    },
    'metrics': {
        'rmse': float(rmse),
        'mae' : float(mae),
        'r2'  : float(r2),
        'mape': float(mape),
    },
    'feature_importance': importance.set_index('feature')['importance'].to_dict(),
    'model_file': config['model']['model_names']['rf_hourly'],
}

with open(result_file, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Results saved: {result_file}")
print("Done!")

Results saved: /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_results_hourly.json
Done!
